# Archived multi-word expression experiments
Historical exploratory notebook, superseded by `notebooks/01_attention_demo.ipynb`. The indentation, dummy tensor conversion, zero-mask handling, and output-directory creation bugs have been patched. Historical paths, setup cells, and execution order are retained; do not use Run All.


In [ ]:

# Now install the required libraries
!pip install daam==0.2.0 diffusers==0.21.2 transformers==4.30.2 torch torchvision matplotlib ftfy scipy
!pip install --upgrade huggingface_hub==0.14.1
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
!pip uninstall diffusers -y
!pip uninstall huggingface_hub -y

In [ ]:
!pip install diffusers==0.21.2
!pip install daam
!pip install huggingface_hub==0.14.1

In [ ]:
from daam import trace, set_seed
from diffusers import DiffusionPipeline
import torch
from matplotlib import pyplot as plt
import numpy as np
import os
import cv2
from datetime import datetime



In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from IPython.display import display


# Initialize the model
model_id = "stabilityai/stable-diffusion-2-1"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to("cuda")

# Generate an image
prompt = "Create a clear and visually engaging timeline illustrating the evolution of text-to-image AI. Highlight key milestones from early AI research to GANs, VAEs, diffusion models, and modern multimodal systems like DALL·E, Midjourney, and Stable Diffusion. Each phase should include a representative image and labels for major breakthroughs, with a clean, minimalistic design."
image = pipe(prompt).images[0]

# Display the image in Colab
display(image)


In [ ]:
from huggingface_hub import login
login()  # Enter your own token through the interactive prompt.


In [ ]:
import daam
print("DAAM version:", daam.__version__)

In [ ]:
import re
import numpy as np
import daam
from matplotlib import pyplot as plt

###############################################################################
# 1. Spatial IoU Computations (Continuous) for MWE Tokens
###############################################################################

def compute_heatmap_iou(heatmap1, heatmap2):
    """
    Compute IoU in a continuous way, ignoring any threshold:
      intersection = sum( min(heatmap1, heatmap2) )
      union        = sum( max(heatmap1, heatmap2) )

    That yields a value between 0 and 1, indicating how similarly
    the two heatmaps overlap in continuous space.
    """
    arr_min = np.minimum(heatmap1, heatmap2)
    arr_max = np.maximum(heatmap1, heatmap2)

    intersection = np.sum(arr_min)
    union = np.sum(arr_max)
    return 0.0 if union == 0 else intersection / union


def compute_mwe_pairwise_iou(heatmaps):
    """
    Compute pairwise IoU between all MWE tokens in `heatmaps` using the
    continuous IoU approach (no threshold).
    Returns a dict: {(token1, token2): iou_value, ...}.
    """
    from itertools import combinations  # Ensure combinations is imported

    pairwise_iou = {}
    mwe_tokens = list(heatmaps.keys())

    for token1, token2 in combinations(mwe_tokens, 2):
        arr1 = heatmaps[token1].heatmap.cpu().numpy()
        arr2 = heatmaps[token2].heatmap.cpu().numpy()
        iou_val = compute_heatmap_iou(arr1, arr2)
        pairwise_iou[(token1, token2)] = iou_val

    return pairwise_iou

###############################################################################
# 2. Display Heatmaps and Binary Masks for MWE Tokens
###############################################################################

class DummyHeatmap:
    """
    If a token doesn't exist in real `heatmaps`, create a dummy zero array.
    """
    def __init__(self, arr):
        self.heatmap = torch.as_tensor(arr)

def display_mwe_heatmaps(output_image, heatmaps, mwe_tokens, title_prefix=""):
    """
    Show the original generated image plus an overlay heatmap for each MWE token.
    """
    n_mwe = len(mwe_tokens)
    cols = 2
    rows = (n_mwe + 1) // cols  # Adjust rows based on number of MWE tokens

    fig, axes = plt.subplots(rows + 1, cols, figsize=(cols * 5, (rows + 1) * 5))
    axes = axes.flatten()

    # First subplot: original image
    axes[0].imshow(output_image)
    axes[0].set_title(f"{title_prefix}Generated Image")
    axes[0].axis("off")

    # Overlay heatmap for each MWE token
    for i, token in enumerate(mwe_tokens):
        idx = i + 1
        if idx >= len(axes):
            break
        ax = axes[idx]
        if token in heatmaps:
            hm_obj = heatmaps[token]
            hm_obj.plot_overlay(output_image, ax=ax)
            ax.set_title(f"{title_prefix}Heatmap: '{token}'")
        else:
            ax.set_title(f"{title_prefix}No heatmap for '{token}'")
        ax.axis("off")

    # Hide any unused subplots
    for j in range(idx + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

def display_mwe_binary_masks(mwe_tokens, heatmaps, threshold_percentile=90):
    """
    For each MWE token, create and display a binary mask based on dynamic thresholding.

    Parameters:
    - mwe_tokens (list): List of MWE tokens.
    - heatmaps (dict): Dictionary of heatmaps for each MWE token.
    - threshold_percentile (float): Percentile to determine the dynamic threshold.
    """
    if not mwe_tokens:
        print("No MWE tokens to display.")
        return

    # Determine shape from existing heatmaps or use default
    shape = None
    if heatmaps:
        shape = next(iter(heatmaps.values())).heatmap.cpu().numpy().shape
    else:
        shape = (64, 64)

    # Build a dict of MWE token -> actual or dummy heatmap
    mwe_heatmaps = {}
    for token in mwe_tokens:
        if token in heatmaps:
            mwe_heatmaps[token] = heatmaps[token]
        else:
            zero_array = np.zeros(shape, dtype=np.float32)
            mwe_heatmaps[token] = DummyHeatmap(zero_array)

    # One subplot per MWE token
    n = len(mwe_tokens)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), squeeze=False)
    axes = axes[0]

    for ax, token in zip(axes, mwe_tokens):
        hm_obj = mwe_heatmaps[token]
        arr = hm_obj.heatmap.cpu().numpy()

        # Normalize heatmap to range [0, 1] if not already
        normalized_heatmap = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

        # Calculate dynamic threshold based on percentile
        dynamic_threshold = np.percentile(normalized_heatmap, threshold_percentile)

        # Create binary heatmap using dynamic threshold
        binary_mask = ((normalized_heatmap > 0) & (normalized_heatmap >= dynamic_threshold)).astype(np.uint8) * 255

        ax.imshow(binary_mask, cmap='gray')
        ax.axis("off")
        ax.set_title(f"Binary Mask: '{token}'", fontsize=12)

    fig.suptitle("Binary Masks for MWE Tokens", fontsize=16, y=1.05)
    plt.tight_layout()
    plt.show()

###############################################################################
# 3. Main Process Prompt (Handle Only MWE)
###############################################################################

def process_prompt(prompt, pipeline, threshold_percentile=90, mwe=None):
    """
    Steps:
      1) Generate an image with DAAM tracing.
      2) Build a heatmap for each MWE token and display them.
      3) Show binary masks for each MWE token using dynamic thresholding.
      4) Compute pairwise IoU (continuous) for MWE tokens.

    Parameters:
    - prompt (str): The input prompt for image generation.
    - pipeline: The diffusion pipeline instance.
    - threshold_percentile (float): Percentile for dynamic thresholding of binary masks.
    - mwe (str): The Multi-Word Expression to focus on.

    Returns:
      A dictionary containing MWE tokens and their pairwise IoU scores.
    """
    if not mwe:
        print("No MWE provided. Exiting process.")
        return

    # 1. Generate the image
    with daam.trace(pipeline) as trc:
        output_image = pipeline(prompt, num_inference_steps=20).images[0]
        global_heat_map = trc.compute_global_heat_map()

    # 2. Get heatmaps for MWE tokens
    mwe_tokens = re.findall(r"\w+", mwe.lower())
    heatmaps = {w: global_heat_map.compute_word_heat_map(w) for w in mwe_tokens}

    # 3. Display overlay heatmaps for MWE tokens
    display_mwe_heatmaps(output_image, heatmaps, mwe_tokens, title_prefix="")

    # 4. Display binary masks for MWE tokens with dynamic thresholding
    display_mwe_binary_masks(
        mwe_tokens,
        heatmaps,
        threshold_percentile=threshold_percentile
    )

    # 5. Compute pairwise IoU for MWE tokens
    if len(mwe_tokens) < 2:
        pairwise_iou = {}
    else:
        pairwise_iou = compute_mwe_pairwise_iou(heatmaps)

    return {
        "Pairwise_IoU": pairwise_iou
    }

###############################################################################
# 4. Example Usage
###############################################################################

if __name__ == "__main__":
    # Ensure 'combinations' is imported for compute_mwe_pairwise_iou
    from itertools import combinations

    # Example usage (replace 'pipe' with your actual diffusion pipeline)
    prompt = "Former NASA astronaut Eugene Cernan dies aged 82 Eugene Cernan, former NASA astronaut and the most recent human to walk on the Moon, died on Monday aged 82, according to a NASA statement."
    mwe = "NASA astronaut"

    # Parameters
    threshold_percentile = 90  # Adjust percentile as needed

    # This threshold_percentile affects MWE binary masks
    scores = process_prompt(
        prompt,
        pipeline=pipe,            # Replace 'pipe' with your actual pipeline instance
        threshold_percentile=threshold_percentile,
        mwe=mwe
    )
    print("\nOverall Scores:", scores)


In [ ]:
import os
import re
import pandas as pd
import numpy as np
import daam
from matplotlib import pyplot as plt
from itertools import combinations
from tqdm import tqdm
import logging
from diffusers import StableDiffusionPipeline  # Ensure diffusers is installed

###############################################################################
# 1. Spatial IoU Computations (Continuous) for MWE Tokens
###############################################################################

def compute_heatmap_iou(heatmap1, heatmap2):
    """
    Compute IoU in a continuous way, ignoring any threshold:
      intersection = sum( min(heatmap1, heatmap2) )
      union        = sum( max(heatmap1, heatmap2) )

    That yields a value between 0 and 1, indicating how similarly
    the two heatmaps overlap in continuous space.
    """
    arr_min = np.minimum(heatmap1, heatmap2)
    arr_max = np.maximum(heatmap1, heatmap2)

    intersection = np.sum(arr_min)
    union = np.sum(arr_max)
    return 0.0 if union == 0 else intersection / union


def compute_mwe_pairwise_iou(heatmaps):
    """
    Compute pairwise IoU between all MWE tokens in heatmaps using the
    continuous IoU approach (no threshold).
    Returns a dict: {(token1, token2): iou_value, ...}.
    """
    pairwise_iou = {}
    mwe_tokens = list(heatmaps.keys())

    for token1, token2 in combinations(mwe_tokens, 2):
        arr1 = heatmaps[token1].heatmap.cpu().numpy()
        arr2 = heatmaps[token2].heatmap.cpu().numpy()
        iou_val = compute_heatmap_iou(arr1, arr2)
        pairwise_iou[(token1, token2)] = iou_val

    return pairwise_iou

###############################################################################
# 2. Display Heatmaps and Binary Masks for MWE Tokens
###############################################################################

class DummyHeatmap:
    """
    If a token doesn't exist in real heatmaps, create a dummy zero array.
    """
    def __init__(self, arr):
        self.heatmap = torch.as_tensor(arr)

def display_mwe_heatmaps(output_image, heatmaps, mwe_tokens, save_path, title_prefix=""):
    """
    Save the original generated image plus an overlay heatmap for each MWE token.

    Parameters:
    - output_image: The generated image.
    - heatmaps: Dictionary of heatmaps.
    - mwe_tokens: List of MWE tokens.
    - save_path: Path to save the heatmap figure.
    - title_prefix: Optional title prefix.
    """
    n_mwe = len(mwe_tokens)
    cols = 2
    rows = (n_mwe + 1) // cols  # Adjust rows based on number of MWE tokens

    fig, axes = plt.subplots(rows + 1, cols, figsize=(cols * 5, (rows + 1) * 5))
    axes = axes.flatten()

    # First subplot: original image
    axes[0].imshow(output_image)
    axes[0].set_title(f"{title_prefix}Generated Image")
    axes[0].axis("off")

    # Overlay heatmap for each MWE token
    for i, token in enumerate(mwe_tokens):
        idx = i + 1
        if idx >= len(axes):
            break
        ax = axes[idx]
        if token in heatmaps:
            hm_obj = heatmaps[token]
            hm_obj.plot_overlay(output_image, ax=ax)
            ax.set_title(f"{title_prefix}Heatmap: '{token}'")
        else:
            ax.set_title(f"{title_prefix}No heatmap for '{token}'")
        ax.axis("off")

    # Hide any unused subplots
    for j in range(idx + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)  # Close the figure to free memory

def display_mwe_binary_masks(mwe_tokens, heatmaps, save_path, threshold_percentile=90):
    """
    Save binary masks for each MWE token based on dynamic thresholding.

    Parameters:
    - mwe_tokens (list): List of MWE tokens.
    - heatmaps (dict): Dictionary of heatmaps for each MWE token.
    - save_path (str): Path to save the binary masks figure.
    - threshold_percentile (float): Percentile to determine the dynamic threshold.
    """
    if not mwe_tokens:
        print("No MWE tokens to display.")
        return

    # Determine shape from existing heatmaps or use default
    shape = None
    if heatmaps:
        shape = next(iter(heatmaps.values())).heatmap.cpu().numpy().shape
    else:
        shape = (64, 64)

    # Build a dict of MWE token -> actual or dummy heatmap
    mwe_heatmaps = {}
    for token in mwe_tokens:
        if token in heatmaps:
            mwe_heatmaps[token] = heatmaps[token]
        else:
            zero_array = np.zeros(shape, dtype=np.float32)
            mwe_heatmaps[token] = DummyHeatmap(zero_array)

    # One subplot per MWE token
    n = len(mwe_tokens)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), squeeze=False)
    axes = axes[0]

    for ax, token in zip(axes, mwe_tokens):
        hm_obj = mwe_heatmaps[token]
        arr = hm_obj.heatmap.cpu().numpy()

        # Normalize heatmap to range [0, 1] if not already
        normalized_heatmap = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

        # Calculate dynamic threshold based on percentile
        dynamic_threshold = np.percentile(normalized_heatmap, threshold_percentile)

        # Create binary heatmap using dynamic threshold
        binary_mask = ((normalized_heatmap > 0) & (normalized_heatmap >= dynamic_threshold)).astype(np.uint8) * 255

        ax.imshow(binary_mask, cmap='gray')
        ax.axis("off")
        ax.set_title(f"Binary Mask: '{token}'", fontsize=12)

    fig.suptitle("Binary Masks for MWE Tokens", fontsize=16, y=1.05)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)  # Close the figure to free memory

###############################################################################
# 3. Main Process Prompt (Handle Only MWE)
###############################################################################

def process_prompt(prompt, pipeline, threshold_percentile=90, mwe=None):
    """
    Steps:
      1) Generate an image with DAAM tracing.
      2) Build a heatmap for each MWE token.
      3) Save binary masks for each MWE token using dynamic thresholding.
      4) Compute pairwise IoU (continuous) for MWE tokens.

    Parameters:
    - prompt (str): The input prompt for image generation.
    - pipeline: The diffusion pipeline instance.
    - threshold_percentile (float): Percentile for dynamic thresholding of binary masks.
    - mwe (str): The Multi-Word Expression to focus on.

    Returns:
      A tuple containing the generated image, heatmaps, and MWE tokens.
    """
    if not mwe:
        print("No MWE provided. Exiting process.")
        return None, None, []

    # 1. Generate the image
    with daam.trace(pipeline) as trc:
        output_image = pipeline(prompt, num_inference_steps=20).images[0]
        global_heat_map = trc.compute_global_heat_map()

    # 2. Get heatmaps for MWE tokens
    mwe_tokens = re.findall(r"\w+", mwe.lower())
    heatmaps = {w: global_heat_map.compute_word_heat_map(w) for w in mwe_tokens}

    return output_image, heatmaps, mwe_tokens

###############################################################################
# 4. Automate Processing from Excel
###############################################################################

def automate_processing_from_excel(excel_path, output_dir, pipeline, threshold_percentile=90, updated_excel_path=None):
    """
    Automate processing of multiple prompts and MWEs from an Excel file.

    Parameters:
    - excel_path (str): Path to the input Excel file.
    - output_dir (str): Directory where output folders will be created.
    - pipeline: The diffusion pipeline instance.
    - threshold_percentile (float): Percentile for dynamic thresholding of binary masks.
    - updated_excel_path (str): Path to save the updated Excel file with IoU scores.
                                 If None, it will be saved as 'processed_output.xlsx' in output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Set default updated_excel_path if not provided
    if updated_excel_path is None:
        updated_excel_path = os.path.join(output_dir, "processed_output.xlsx")

    # Set up logging
    log_file_path = os.path.join(output_dir, 'processing.log')
    logging.basicConfig(filename=log_file_path, level=logging.INFO,
                        format='%(asctime)s - %(levelname)s - %(message)s')

    # Read Excel file
    try:
        df = pd.read_excel(excel_path)
        logging.info(f"Successfully read Excel file: {excel_path}")
    except Exception as e:
        logging.error(f"Failed to read Excel file: {excel_path}. Error: {e}")
        raise

    # Check required columns
    required_columns = ['ai_sorted', 'Expression']  # Updated column name here
    for col in required_columns:
        if col not in df.columns:
            error_msg = f"Excel file must contain '{col}' column."
            logging.error(error_msg)
            raise ValueError(error_msg)

    # Prepare to store IoU scores
    iou_scores = []

    # Iterate over each row with a progress bar
    for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing rows"):
        prompt = row['ai_sorted']  # Updated column name here
        mwe = row['Expression']

        # Extract MWE tokens
        mwe_tokens = re.findall(r"\w+", mwe.lower())

        # Create folder name: index_number_mwe_tokens (replace spaces with underscores)
        folder_name = f"{index}_{'_'.join(mwe_tokens)}"
        folder_path = os.path.join(output_dir, folder_name)
        os.makedirs(folder_path, exist_ok=True)

        logging.info(f"Processing row {index}: MWE='{mwe}'")

        try:
            # Process prompt
            output_image, heatmaps, mwe_tokens = process_prompt(
                prompt=prompt,
                pipeline=pipeline,
                threshold_percentile=threshold_percentile,
                mwe=mwe
            )

            if output_image is None:
                raise ValueError("MWE is empty.")

            # Save heatmap overlays
            heatmap_save_path = os.path.join(folder_path, "heatmaps.png")
            display_mwe_heatmaps(
                output_image=output_image,
                heatmaps=heatmaps,
                mwe_tokens=mwe_tokens,
                save_path=heatmap_save_path,
                title_prefix=""
            )

            # Save binary masks
            binary_mask_save_path = os.path.join(folder_path, "binary_masks.png")
            display_mwe_binary_masks(
                mwe_tokens=mwe_tokens,
                heatmaps=heatmaps,
                save_path=binary_mask_save_path,
                threshold_percentile=threshold_percentile
            )

            # Compute pairwise IoU
            if len(mwe_tokens) < 2:
                pairwise_iou = {}
            else:
                pairwise_iou = compute_mwe_pairwise_iou(heatmaps)

            # Save IoU scores to txt file
            iou_txt_path = os.path.join(folder_path, "iou_scores.txt")
            with open(iou_txt_path, 'w') as f:
                for pair, score in pairwise_iou.items():
                    f.write(f"{pair}: {score:.4f}\n")  # Format to 4 decimal places

            # Determine the maximum IoU score
            if pairwise_iou:
                max_iou = max(pairwise_iou.values())
            else:
                max_iou = np.nan  # or 0.0, depending on preference

            iou_scores.append(round(max_iou, 4))  # Round to 4 decimals
            logging.info(f"Row {index} processed successfully. Max IoU: {max_iou}")

        except Exception as e:
            logging.error(f"Error processing row {index}: {e}")
            iou_scores.append(np.nan)  # Assign NaN for failed rows
            continue  # Proceed to the next row

    # Add IoU scores to DataFrame
    df['Max_IoU'] = iou_scores

    # Save updated Excel
    try:
        df.to_excel(updated_excel_path, index=False)
        logging.info(f"Processing complete. Updated Excel saved to {updated_excel_path}")
        print(f"Processing complete. Updated Excel saved to {updated_excel_path}")
    except Exception as e:
        logging.error(f"Failed to save updated Excel file to {updated_excel_path}. Error: {e}")
        raise


###############################################################################
# 5. Example Usage
###############################################################################

if __name__ == "__main__":
    # ---------------------------------------------
    # 1. Specify the Path to Your Input Excel File
    # ---------------------------------------------
    # Replace the string below with the path to your Excel file.
    # Example: "C:/Users/YourName/Documents/input_prompts.xlsx"
    input_excel_path = "/content/drive/MyDrive/Thesis/dataset/verb_particle/sorted-verb_particle.xlsx"  # <-- Replace with your Excel file path

    # ---------------------------------------------
    # 2. Specify the Output Directory
    # ---------------------------------------------
    # Replace the string below with the desired output directory path.
    # Example: "C:/Users/YourName/Documents/output_results"
    output_directory = "/content/drive/MyDrive/Thesis/dataset/verb_particle/images/"  # <-- Replace with your desired output directory
    os.makedirs(output_directory, exist_ok=True)  # Creates the directory if it doesn't exist

    # ---------------------------------------------
    # 3. Specify the Path for the Updated Excel File
    # ---------------------------------------------
    # Replace the string below with the desired path for the updated Excel file.
    # If you want it to be inside the output directory, you can specify it accordingly.
    # Example: "C:/Users/YourName/Documents/output_results/processed_output.xlsx"
    updated_excel_output_path = os.path.join(output_directory, "processed_output.xlsx")  # <-- Replace if needed

    # ---------------------------------------------
    # 4. Initialize Your Diffusion Pipeline
    # ---------------------------------------------
    # Ensure you have the 'diffusers' library installed:
    # pip install diffusers transformers accelerate

    # Replace the string below with the model you intend to use.
    # You can use other models available on Hugging Face's model hub.
    # Example: "runwayml/stable-diffusion-v1-5"

    try:
        print("Initializing the diffusion pipeline...")
        diffusion_model_name = "stabilityai/stable-diffusion-2-1"  # <-- Replace with your model
        pipe = StableDiffusionPipeline.from_pretrained(diffusion_model_name)
        pipe = pipe.to("cuda")  # Use "cuda" if you have a GPU, else "cpu"
        print("Diffusion pipeline initialized successfully.")
    except Exception as e:
        print(f"Error initializing the diffusion pipeline: {e}")
        raise

    # ---------------------------------------------
    # 5. Set Threshold Percentile (Optional)
    # ---------------------------------------------
    # Adjust this value as needed for dynamic thresholding of binary masks.
    threshold_percentile_value = 90  # <-- Adjust as desired

    # ---------------------------------------------
    # 6. Automate Processing
    # ---------------------------------------------
    try:
        automate_processing_from_excel(
            excel_path=input_excel_path,
            output_dir=output_directory,
            pipeline=pipe,
            threshold_percentile=threshold_percentile_value,
            updated_excel_path=updated_excel_output_path
        )
    except Exception as e:
        print(f"An error occurred during processing: {e}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')